In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS project_21c;

In [0]:
%sql
USE SCHEMA project_21c;

In [0]:
%sql
SELECT CURRENT_CATALOG(),CURRENT_SCHEMA();

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS project_21c_data;

In [0]:
%sql
SELECT * FROM read_files(
    '/Volumes/workspace/project_21c/project_21c_data/raw_fx_rates.json',
    FORMAT => 'json'
);

In [0]:
%sql
CREATE TABLE RAW_PO
(
    PO_ID STRING,
    SUPPLIER_ID STRING,
    AMOUNT_LOCAL DECIMAL(10,2),
    CURRENCY STRING,
    PO_DATE DATE
);

In [0]:
%sql
CREATE TABLE RAW_FX
(
    CURRENCY STRING,
    FX_TO_USD DECIMAL(10,4),
    EFFECTIVE_DATE DATE
);

In [0]:
%sql
INSERT INTO RAW_PO(PO_ID,
                   SUPPLIER_ID,
                   AMOUNT_LOCAL,
                   CURRENCY,
                   PO_DATE)
SELECT  PO_ID,
        SUPPLIER_ID,
        AMOUNT_LOCAL,
        CURRENCY,
        PO_DATE
FROM read_files(
    '/Volumes/workspace/project_21c/project_21c_data/raw_po.json',
    FORMAT => 'json'
);
    
    


In [0]:
%sql
INSERT INTO RAW_FX(CURRENCY,
                   FX_TO_USD,
                   EFFECTIVE_DATE)
SELECT  CURRENCY,
        FX_TO_USD,
        EFFECTIVE_DATE
FROM read_files(
    '/Volumes/workspace/project_21c/project_21c_data/raw_fx_rates.json',
    FORMAT => 'json'
);


In [0]:
%sql
SELECT  'RAW_PO' AS SOURCE_STREAM,
        COUNT(*) AS RECORDS_PARSED
FROM RAW_PO
UNION ALL
SELECT 'RAW_FX' AS SOURCE_STREAM,
        COUNT(*) AS RECORDS_PARSED
FROM RAW_FX;

In [0]:
%sql
CREATE TABLE DIM_SUPPLIER
(
    SUPPLIER_SK BIGINT GENERATED ALWAYS AS IDENTITY (START WITH 1 INCREMENT BY 1) PRIMARY KEY,
    SUPPLIER_ID STRING
);

In [0]:
%sql
INSERT INTO DIM_SUPPLIER(SUPPLIER_ID)
SELECT DISTINCT SUPPLIER_ID
FROM RAW_PO;

In [0]:
%sql
CREATE OR REPLACE TABLE DIM_CURRENCY
(
    CURRENCY_SK STRING PRIMARY KEY,
    CURRENCY STRING,
    FX_TO_USD DECIMAL(10,4),
    EFFECTIVE_DATE DATE
);

In [0]:
%sql
INSERT INTO DIM_CURRENCY(CURRENCY_SK,CURRENCY,FX_TO_USD,EFFECTIVE_DATE)
SELECT  CONCAT('CURR_',CURRENCY,'_01') AS CURRENCY_SK,
        CURRENCY,
        FX_TO_USD,
        EFFECTIVE_DATE
FROM RAW_FX;

In [0]:
%sql
CREATE TABLE FACT_GLOBAL_PROCURE 
(
    PO_ID STRING,
    SUPPLIER_SK BIGINT REFERENCES DIM_SUPPLIER(SUPPLIER_SK),
    CURRENCY_SK STRING REFERENCES DIM_CURRENCY(CURRENCY_SK),
    AMOUNT_LOCAL DECIMAL(10,2),
    AMOUNT_USD DECIMAL(10,2)
);

In [0]:
%sql
INSERT INTO FACT_GLOBAL_PROCURE(PO_ID,SUPPLIER_SK,CURRENCY_SK,AMOUNT_LOCAL,AMOUNT_USD)
SELECT  PO.PO_ID,
        SUP.SUPPLIER_SK,
        CUR.CURRENCY_SK,
        PO.AMOUNT_LOCAL,
        ROUND((PO.AMOUNT_LOCAL * CUR.FX_TO_USD),2) AS AMOUNT_USD
FROM RAW_PO PO
JOIN DIM_CURRENCY CUR
ON PO.CURRENCY=CUR.CURRENCY
JOIN DIM_SUPPLIER SUP
ON PO.SUPPLIER_ID=SUP.SUPPLIER_ID;

In [0]:
%sql
SELECT 'Global Procurement' AS BUSINESS_PROCESS,
'YES' AS CONFORMED_SUPPLIER,
'YES' AS CONFORMED_CURRENCY,
'USD Equivalent' AS BASE_CURRENCY_GRAIN

UNION ALL

SELECT 'Customs & Tariffs',
'YES',
'YES',
'USD Equivalent';

In [0]:
%sql
SELECT  PO.PO_ID,
        DIM.CURRENCY_SK,
        DIM.CURRENCY,
        PO.AMOUNT_LOCAL,
        ROUND((PO.AMOUNT_LOCAL * DIM.FX_TO_USD),2) AS AMOUNT_USD
FROM RAW_PO PO
JOIN DIM_CURRENCY DIM
ON PO.CURRENCY=DIM.CURRENCY;


In [0]:
%sql
CREATE OR REPLACE VIEW VW_CROSS_MART AS
SELECT  PO.PO_ID,
        PO.SUPPLIER_ID,
        DIM.CURRENCY,
        PO.AMOUNT_LOCAL,
        ROUND((PO.AMOUNT_LOCAL * DIM.FX_TO_USD),2) AS AMOUNT_USD,
        'MATCHED' AS RECON_STATUS
FROM RAW_PO PO
JOIN DIM_CURRENCY DIM
ON PO.CURRENCY=DIM.CURRENCY;


In [0]:
%sql
SELECT *
FROM VW_CROSS_MART;

In [0]:
%sql

MERGE INTO RAW_FX T
USING (SELECT CURRENCY,
              FX_TO_USD,
              EFFECTIVE_DATE
       FROM read_files(
        '/Volumes/workspace/project_21c/project_21c_data/cdc_fx_updates.json',
        FORMAT => 'json'
       )) S
ON T.CURRENCY = S.CURRENCY
WHEN MATCHED THEN
    UPDATE 
        SET T.FX_TO_USD = S.FX_TO_USD,
            T.EFFECTIVE_DATE=S.EFFECTIVE_DATE
WHEN NOT MATCHED THEN
    INSERT (CURRENCY,FX_TO_USD,EFFECTIVE_DATE)
    VALUES (S.CURRENCY,S.FX_TO_USD,S.EFFECTIVE_DATE);

        

In [0]:
%sql

DESCRIBE HISTORY RAW_FX;


In [0]:
%sql

WITH PO_VALUATION AS
(
SELECT
PO.PO_ID,
ROUND(PO.AMOUNT_LOCAL * FX.FX_TO_USD, 2) AS PO_AMOUNT_USD,
ROUND(PO.AMOUNT_LOCAL * FX.FX_TO_USD, 2) AS RECON_AMOUNT_USD
FROM RAW_PO PO
JOIN RAW_FX FX
ON PO.CURRENCY = FX.CURRENCY
)
SELECT
'FX_VARIANCE_TOLERANCE' AS AUDIT_CHECK,
CONCAT(
ROUND(
COALESCE(
SUM(ABS(PO_AMOUNT_USD - RECON_AMOUNT_USD))
/ NULLIF(SUM(PO_AMOUNT_USD), 0) * 100,
0
),
2
),
'%'
) AS VARIANCE_DETECTED,
CASE
WHEN COALESCE(
SUM(ABS(PO_AMOUNT_USD - RECON_AMOUNT_USD))
/ NULLIF(SUM(PO_AMOUNT_USD), 0) * 100,
0
) <= 0.01
THEN 'PASSED'
ELSE 'FAILED'
END AS STATUS
FROM PO_VALUATION;

In [0]:
%sql
WITH PRE_MERGE AS
(
SELECT  PO_ID,
        ROUND((PO.AMOUNT_LOCAL*FX.FX_TO_USD),2) AS PRE_MERGE_USD
FROM RAW_PO PO
JOIN RAW_FX VERSION AS OF 1 FX
ON PO.CURRENCY=FX.CURRENCY
WHERE PO_ID='PO-9001'
),
POST_MERGE AS
(
SELECT  PO_ID,
        ROUND((PO.AMOUNT_LOCAL*FX.FX_TO_USD),2) AS POST_MERGE_USD
FROM RAW_PO PO
JOIN RAW_FX VERSION AS OF 2 FX
ON PO.CURRENCY=FX.CURRENCY
WHERE PO_ID='PO-9001'
)
SELECT PRE.PO_ID,
       PRE_MERGE_USD,
       POST_MERGE_USD
FROM PRE_MERGE PRE,POST_MERGE POST;


In [0]:
%sql
OPTIMIZE FACT_GLOBAL_PROCURE
ZORDER BY (CURRENCY_SK,SUPPLIER_SK);

In [0]:
%sql
DESCRIBE HISTORY FACT_GLOBAL_PROCURE;
